In [ ]:
# Fabric Notebook — 03_gold_curate
# Silver (cleansed) -> Gold (star schema: dims, facts, bridge)
#
# Builds the dimensional model the semantic model / Data Agent sits on:
#   dim_farmer, dim_product, dim_plant, dim_customer, dim_date, dim_test_type
#   fact_intake, fact_production_run, fact_quality_test, fact_sales_order
#   bridge_lot_run

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ---- Config: lakehouse / schema ----
# Source and target live in the same lakehouse, different schemas.
LAKEHOUSE = "agri_lakehouse"
SILVER_SCHEMA = "silverdb"   # source schema
GOLD_SCHEMA = "golddb"       # target schema
TABLE_SUFFIX = "_dim"        # both silver and gold tables use the <name>_dim convention

def silver(name):
    return spark.read.table(f"{LAKEHOUSE}.{SILVER_SCHEMA}.{name}{TABLE_SUFFIX}")

def gold(name):
    return spark.read.table(f"{LAKEHOUSE}.{GOLD_SCHEMA}.{name}{TABLE_SUFFIX}")

def write_gold(df, name):
    target = f"{LAKEHOUSE}.{GOLD_SCHEMA}.{name}{TABLE_SUFFIX}"
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(target))
    print(f"  wrote {target}: {df.count():,} rows")

# Load silver tables
s_farmers     = silver("farmers")
s_lots        = silver("lots")
s_runs        = silver("production_runs")
s_consumption = silver("lot_consumption")
s_quality     = silver("quality_tests")
s_sales       = silver("sales_orders")

print("=== GOLD LAYER ===")

# -------------------------------------------------------------------
# DIMENSIONS
# -------------------------------------------------------------------

# --- dim_farmer ---
print("Building dim_farmer...")
dim_farmer = (
    s_farmers
    .select(
        F.col("farmer_id").alias("farmer_key"),
        "farmer_id",
        "name",
        "district",
        F.col("district").alias("province"),   # province == district label here
        "organic_certification",
        "is_organic",
    )
)
write_gold(dim_farmer, "farmer")

# --- dim_product (derived from runs) ---
print("Building dim_product...")
dim_product = (
    s_runs.select("product", F.col("is_organic_run").alias("is_organic"))
    .distinct()
    # Derive a product family from the SKU name
    .withColumn("product_family",
        F.when(F.col("product").contains("Coconut Water"), F.lit("Coconut Water"))
         .when(F.col("product").contains("Coconut Milk"),  F.lit("Coconut Milk"))
         .when(F.col("product").contains("Coconut Cream"), F.lit("Coconut Cream"))
         .when(F.col("product").contains("Coconut Oil"),   F.lit("Coconut Oil"))
         .when(F.col("product").contains("Desiccated"),    F.lit("Desiccated Coconut"))
         .otherwise(F.lit("Other")))
    .withColumn("product_key", F.sha2(F.concat_ws("||", "product"), 256))
    .select("product_key", "product", "product_family", "is_organic")
)
write_gold(dim_product, "product")

# --- dim_plant (derived from runs: plant + line) ---
print("Building dim_plant...")
dim_plant = (
    s_runs.select("plant", "line").distinct()
    .withColumn("plant_key", F.sha2(F.concat_ws("||", "plant", "line"), 256))
    .withColumn("plant_type",
        F.when(F.col("plant").contains("Organic"), F.lit("Organic"))
         .otherwise(F.lit("Conventional")))
    .select("plant_key", "plant", "line", "plant_type")
)
write_gold(dim_plant, "plant")

# --- dim_customer (derived from sales) ---
print("Building dim_customer...")
dim_customer = (
    s_sales.select("customer", "country", "region", "channel", "is_domestic").distinct()
    .withColumn("customer_key",
                F.sha2(F.concat_ws("||", "customer", "country", "channel"), 256))
    .select("customer_key", "customer", "country", "region", "channel", "is_domestic")
)
write_gold(dim_customer, "customer")

# --- dim_date (generated to span all dates in the data) ---
print("Building dim_date...")
# Find the min/max dates across all date columns
bounds = (
    s_lots.select(F.col("intake_date").alias("d"))
    .union(s_runs.select(F.col("run_date").alias("d")))
    .union(s_sales.select(F.col("order_date").alias("d")))
    .union(s_quality.select(F.col("test_date").alias("d")))
    .agg(F.min("d").alias("min_d"), F.max("d").alias("max_d"))
    .collect()[0]
)
min_d, max_d = bounds["min_d"], bounds["max_d"]
dim_date = (
    spark.sql(f"SELECT explode(sequence(to_date('{min_d}'), to_date('{max_d}'), interval 1 day)) AS date")
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.concat(F.lit("Q"), F.quarter("date")))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_of_week", F.date_format("date", "EEEE"))
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn("is_weekend", F.dayofweek("date").isin(1, 7))
    .select("date_key", "date", "year", "quarter", "month", "month_name",
            "day", "day_of_week", "week_of_year", "is_weekend")
)
write_gold(dim_date, "date")

# --- dim_test_type (derived from quality tests) ---
print("Building dim_test_type...")
dim_test_type = (
    s_quality.select("test_name", "category", "target", "tolerance", "unit").distinct()
    .withColumn("test_type_key", F.sha2(F.col("test_name"), 256))
    .select("test_type_key", "test_name", "category", "target", "tolerance", "unit")
)
write_gold(dim_test_type, "test_type")

# -------------------------------------------------------------------
# FACTS
# -------------------------------------------------------------------

# --- fact_intake (one row per lot) ---
print("Building fact_intake...")
fact_intake = (
    s_lots
    .withColumn("date_key", F.date_format("intake_date", "yyyyMMdd").cast("int"))
    .select(
        "lot_id",
        F.col("farmer_id").alias("farmer_key"),
        "date_key",
        "intake_date",
        "collection_point",
        "district",
        "weight_kg",
        "brix",
        "grade",
        "is_organic",
        "quality_flag",
    )
)
write_gold(fact_intake, "fact_intake")

# --- fact_production_run (one row per run) ---
print("Building fact_production_run...")
fact_run = (
    s_runs
    .withColumn("date_key", F.date_format("run_date", "yyyyMMdd").cast("int"))
    .withColumn("product_key", F.sha2(F.concat_ws("||", "product"), 256))
    .withColumn("plant_key", F.sha2(F.concat_ws("||", "plant", "line"), 256))
    .withColumn("yield_loss",
                F.round(F.col("expected_yield") - F.col("actual_yield"), 1))
    .select(
        "run_id", "date_key", "product_key", "plant_key",
        "plant", "line", "product", "shift",
        "start_datetime", "end_datetime", "duration_hours",
        "input_weight_kg", "expected_yield", "actual_yield",
        "yield_pct", "yield_loss", "yield_band",
        "avg_brix", "grade_a_pct", "downtime_minutes",
        "is_organic_run", "num_lots_consumed",
    )
)
write_gold(fact_run, "fact_production_run")

# --- fact_quality_test (one row per test) ---
print("Building fact_quality_test...")
fact_quality = (
    s_quality
    .withColumn("date_key", F.date_format("test_date", "yyyyMMdd").cast("int"))
    .withColumn("test_type_key", F.sha2(F.col("test_name"), 256))
    .select(
        "test_id", "run_id", "test_type_key", "date_key",
        "test_name", "category", "value", "target", "tolerance", "unit",
        "pass_fail", "test_datetime",
    )
)
write_gold(fact_quality, "fact_quality_test")

# --- fact_sales_order (one row per order) ---
print("Building fact_sales_order...")
fact_sales = (
    s_sales
    .withColumn("date_key", F.date_format("order_date", "yyyyMMdd").cast("int"))
    .withColumn("product_key", F.sha2(F.concat_ws("||", "product"), 256))
    .withColumn("customer_key",
                F.sha2(F.concat_ws("||", "customer", "country", "channel"), 256))
    .select(
        "order_id", "date_key", "product_key", "customer_key", "run_id",
        "order_date", "customer", "country", "region", "channel",
        "is_domestic", "is_organic", "product",
        "quantity_units", "unit_price_usd",
        "revenue_usd", "revenue_local", "currency",
        "margin_pct", "margin_usd", "on_time_delivery",
    )
)
write_gold(fact_sales, "fact_sales_order")

# --- bridge_lot_run (lot <-> run, with farmer for fast traceability) ---
print("Building bridge_lot_run...")
bridge = (
    s_consumption
    .select(
        "run_id",
        "lot_id",
        F.col("farmer_id").alias("farmer_key"),
        "weight_consumed",
    )
)
write_gold(bridge, "bridge_lot_run")

print("\n=== GOLD COMPLETE ===")

# -------------------------------------------------------------------
# OPTIONAL: quick validation / smoke test
# -------------------------------------------------------------------
print("\n--- Validation ---")
print(f"farmer:          {gold('farmer').count():,}")
print(f"fact_production_run: {gold('fact_production_run').count():,}")
print(f"fact_sales_order:    {gold('fact_sales_order').count():,}")

# Sanity: average yield by plant (the headline KPI)
(gold("fact_production_run")
   .groupBy("plant")
   .agg(F.round(F.avg("yield_pct"), 2).alias("avg_yield_pct"),
        F.count("*").alias("runs"))
   .orderBy("plant")
   .show(truncate=False))

# Sanity: end-to-end traceability — one export order back to farmers
print("Traceability sample (one order -> run -> lots -> farmers):")
sample_order = (gold("fact_sales_order")
                .filter(~F.col("is_domestic")).limit(1).collect()[0])
sample_run = sample_order["run_id"]
(gold("bridge_lot_run")
   .filter(F.col("run_id") == sample_run)
   .join(gold("farmer"), "farmer_key")
   .select("run_id", "lot_id", "name", "district")
   .show(10, truncate=False))